# 实验二　房源采集与 CSV 数据存储

- 数据来源：`http://www.bspider.top/anjuke/`（安居客教学站）
- 采集字段：title → house → address → struct → area → floor → maketime → price
- 输出文件：`data/houseinfo.csv`（utf-8-sig 编码）

本 Notebook 由三部分组成：① 请求并检查第一页；② 分析选择器、解析 8 个字段；③ 翻页采集、去重并保存 CSV。


In [1]:
import csv, time, re
from pathlib import Path
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

BASE_URL = "http://www.bspider.top/anjuke/"
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
CSV_PATH = DATA_DIR / "houseinfo.csv"

FIELDS = ["title", "house", "address", "struct", "area", "floor", "maketime", "price"]
print("字段顺序：", FIELDS)


字段顺序： ['title', 'house', 'address', 'struct', 'area', 'floor', 'maketime', 'price']


## 1. 获取第一页并检查响应

先请求一次，输出状态码、正文长度和页面标题。状态码 200 也要看标题，确认拿到的是房源列表而不是访问验证页。


In [2]:
r = requests.get(BASE_URL, headers=HEADERS, timeout=30)
r.raise_for_status()
r.encoding = r.apparent_encoding
soup = BeautifulSoup(r.text, "html.parser")

print("status:", r.status_code)
print("长度:", len(r.text))
print("标题:", soup.title.get_text(strip=True))


status: 200
长度: 280427
标题: 重庆二手房房产网，重庆二手房出售，重庆买房购房交易信息 - 58安居客


## 2. 选择器：先定位“一条房源”的父容器

在 Chrome 中按 F12 → Elements → 选取房源标题 → 右键 Copy → Copy selector，得到的是从页面根节点开始的完整路径，
中间夹着多层 `div:nth-child()`，页面结构一变就失效。

向上找一层刚好包住整条房源的节点（同时含标题、楼盘、地址、属性、价格），可见它的 class 是 `property`。
所以把“整页长路径”改成两步：先 `select("div.property")` 拿到全部房源，再在每条房源内部用相对选择器取字段。


In [3]:
rows = soup.select("div.property")
print("本页房源条数：", len(rows))

# 打印第一条房源的容器 class 与标题，确认父容器刚好包含一条房源
first = rows[0]
print("父容器标签/类：", first.name, first.get("class"))
print("容器内 h3 个数：", len(first.select("h3")))
print("第一条标题：", first.select_one("h3").get_text(strip=True))


本页房源条数： 60
父容器标签/类： div ['property']
容器内 h3 个数： 1
第一条标题： 万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器


### 拆解一条房源里的 8 个字段

先在第一条上逐个字段试取值，特别确认**属性组里 4 个 `<p>` 的顺序**（面积、朝向、楼层、建造年代），
避免把“南北”当成楼层、把“总价”当成单价。


In [4]:
row = rows[0]

# 标题
print("title  :", row.select_one("h3").get_text(strip=True))
# 楼盘
print("house  :", row.select_one("p.property-content-info-comm-name").get_text(strip=True))
# 地址：文字分散在 3 个 span 里，直接取整段文字，不额外插空格
addr = row.select_one("p.property-content-info-comm-address")
print("address:", addr.get_text(strip=True), "| span 个数:", len(addr.select("span")))
# 户型
print("struct :", row.select_one("p.property-content-info-attribute").get_text(strip=True))

# 属性组：逐项打印，确认顺序
info = row.select("div.property-content-info")[0]
items = [p.get_text(strip=True) for p in info.select("p.property-content-info-text")]
for i, t in enumerate(items):
    print(f"  属性[{i}] = {t}")

# 单价（每平方米），不是总价“148 万”
print("price  :", row.select_one("p.property-price-average").get_text(strip=True))
print("总价（不采集）:", row.select_one("p.property-price-total").get_text(strip=True).replace("\n", " "))


title  : 万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器
house  : 万科金域学府翰林菁华里
address: 南岸学府大道二塘路59号 | span 个数: 3
struct : 3室2厅1卫
  属性[0] = 3室2厅1卫
  属性[1] = 89.9㎡
  属性[2] = 南北
  属性[3] = 低层(共27层)
  属性[4] = 2019年建造
price  : 16465元/㎡
总价（不采集）: 148万


## 3. 编写解析函数

属性组的顺序在不同房源里可能变化，也可能缺少“建造年代”，所以**不按固定下标取值**，改为按内容特征判断：
含 `㎡` 的是面积，含 `层` 的是楼层，含 `年建造` 的是建造年代，其余是朝向（不采集）。
建造年代缺失时填空字符串，不让整条记录失败。


In [5]:
def parse_record(row):
    """按 title→house→address→struct→area→floor→maketime→price 的顺序返回一条记录。"""

    def text_of(selector):
        node = row.select_one(selector)
        return node.get_text(strip=True) if node else ""

    title = text_of("h3")
    house = text_of("p.property-content-info-comm-name")
    address = text_of("p.property-content-info-comm-address")
    struct = text_of("p.property-content-info-attribute")

    area = floor = maketime = ""
    info = row.select_one("div.property-content-info")
    if info is not None:
        for p in info.select("p.property-content-info-text"):
            t = p.get_text(strip=True)
            if "㎡" in t or "平米" in t:
                area = t
            elif "层" in t:
                floor = t
            elif "年" in t and "建造" in t:
                maketime = t
    # 建造年代节点缺失时保持空字符串
    if not maketime:
        maketime = ""

    price = text_of("p.property-price-average")
    return [title, house, address, struct, area, floor, maketime, price]


rec = parse_record(rows[0])
for k, v in zip(FIELDS, rec):
    print(f"{k:9s}= {v}")
print("字段个数：", len(rec))


title    = 万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器
house    = 万科金域学府翰林菁华里
address  = 南岸学府大道二塘路59号
struct   = 3室2厅1卫
area     = 89.9㎡
floor    = 低层(共27层)
maketime = 2019年建造
price    = 16465元/㎡
字段个数： 8


## 4. 下一页：取 href、用 urljoin 补全

“下一页”按钮是 `a.next`，可点时多一个 `next-active` 类；末页按钮的 href 变成 `#` 且 class 只剩 `next`。
这里的 href 只有 `?page=2`，是相对地址，必须用 `urljoin(current_url, href)` 拼成完整地址。


In [6]:
next_a = soup.select_one("a.next.next-active")
href = next_a.get("href") if next_a else None
print("下一页 href 原值：", href)
print("urljoin 补全后 ：", urljoin(BASE_URL, href))

# 末页的样子（第 10 页）
r10 = requests.get(BASE_URL + "?page=10", headers=HEADERS, timeout=30)
soup10 = BeautifulSoup(r10.text, "html.parser")
a10 = soup10.select_one("a.next")
print("第10页 a.next 的 class：", a10.get("class"), "href：", a10.get("href"))
print("第10页房源条数：", len(soup10.select("div.property")))


下一页 href 原值： ?page=2
urljoin 补全后 ： http://www.bspider.top/anjuke/?page=2


第10页 a.next 的 class： ['next'] href： #
第10页房源条数： 60


## 5. 分页采集

循环里做四件事：解析本页 → 找下一页并补全 → 检查地址是否越界/重复 → 等待 1 秒再请求。
停止条件：`a.next.next-active` 不存在（末页按钮被禁用）。
另外设 `MAX_PAGES` 上限和“已访问地址集合”，防止无限循环和重复采集。


In [7]:
def collect(max_pages=30, delay=1.0):
    records = []
    visited = set()
    current_url = BASE_URL
    page_no = 0
    stop_reason = ""

    while page_no < max_pages:
        if current_url in visited:
            stop_reason = "地址重复，停止：" + current_url
            break
        visited.add(current_url)

        r = requests.get(current_url, headers=HEADERS, timeout=30)
        r.raise_for_status()
        r.encoding = r.apparent_encoding
        page_soup = BeautifulSoup(r.text, "html.parser")

        rows = page_soup.select("div.property")
        page_no += 1
        if not rows:
            stop_reason = "本页房源为 0 条，判定为异常结构，停止。"
            print(stop_reason)
            break

        for row in rows:
            records.append(parse_record(row))
        print(f"第{page_no:2d}页  URL={current_url}  本页={len(rows):2d}条  累计={len(records)}条")

        next_a = page_soup.select_one("a.next.next-active")
        if next_a is None:
            stop_reason = f"第{page_no}页“下一页”按钮已禁用（a.next 无 next-active），判定到达末页。"
            break

        next_url = urljoin(current_url, next_a.get("href"))
        if not next_url.startswith(BASE_URL):
            stop_reason = "下一页地址越出教学站，停止：" + next_url
            break
        current_url = next_url
        time.sleep(delay)
    else:
        stop_reason = f"达到安全页数上限 {max_pages} 页，中止（不算采集完成）。"

    return records, page_no, stop_reason


all_records, pages, stop_reason = collect(max_pages=30, delay=1.0)
print("-" * 60)
print("成功采集页数：", pages)
print("采集记录数（去重前）：", len(all_records))
print("停止原因：", stop_reason)


第 1页  URL=http://www.bspider.top/anjuke/  本页=60条  累计=60条


第 2页  URL=http://www.bspider.top/anjuke/?page=2  本页=60条  累计=120条


第 3页  URL=http://www.bspider.top/anjuke/?page=3  本页=60条  累计=180条


第 4页  URL=http://www.bspider.top/anjuke/?page=4  本页=60条  累计=240条


第 5页  URL=http://www.bspider.top/anjuke/?page=5  本页=60条  累计=300条


第 6页  URL=http://www.bspider.top/anjuke/?page=6  本页=60条  累计=360条


第 7页  URL=http://www.bspider.top/anjuke/?page=7  本页=60条  累计=420条


第 8页  URL=http://www.bspider.top/anjuke/?page=8  本页=60条  累计=480条


第 9页  URL=http://www.bspider.top/anjuke/?page=9  本页=60条  累计=540条


第10页  URL=http://www.bspider.top/anjuke/?page=10  本页=60条  累计=600条
------------------------------------------------------------
成功采集页数： 10
采集记录数（去重前）： 600
停止原因： 第10页“下一页”按钮已禁用（a.next 无 next-active），判定到达末页。


## 6. 去重并保存 CSV

按 8 个字段完全相同去重：用 `dict.fromkeys` 保持原顺序，`tuple` 可哈希。
文件用 `utf-8-sig`（Excel 打开不乱码）和 `newline=""`（避免空行）。


In [8]:
unique = list(dict.fromkeys(tuple(rec) for rec in all_records))
dup = len(all_records) - len(unique)
print("去重前：", len(all_records), " 完全相同的重复记录：", dup, " 去重后：", len(unique))

with open(CSV_PATH, "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(FIELDS)
    for rec in unique:
        assert len(rec) == 8, rec
        writer.writerow(rec)
print("已写入：", CSV_PATH.resolve())


去重前： 600  完全相同的重复记录： 102  去重后： 498
已写入： /home/pwuser/workspace/lab02/data/houseinfo.csv


### 回读核对

用 `csv.DictReader` 重新读文件，统计数据行数（表头不计），并打印首条、跨页一条和最后一条。


In [9]:
with open(CSV_PATH, encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    print("表头：", reader.fieldnames)
    data = list(reader)

print("CSV 数据行数（不含表头）：", len(data))
print()
for pos in (0, 60, len(data) - 1):
    rec = data[pos]
    print(f"--- 第 {pos + 1} 行 ---")
    for k in FIELDS:
        print(f"  {k:9s}= {rec[k]}")


表头： ['title', 'house', 'address', 'struct', 'area', 'floor', 'maketime', 'price']
CSV 数据行数（不含表头）： 498

--- 第 1 行 ---
  title    = 万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器
  house    = 万科金域学府翰林菁华里
  address  = 南岸学府大道二塘路59号
  struct   = 3室2厅1卫
  area     = 89.9㎡
  floor    = 低层(共27层)
  maketime = 2019年建造
  price    = 16465元/㎡
--- 第 61 行 ---
  title    = 万科物业自营 朝中庭 业主诚心出售
  house    = 万科金域学府翰林菁华里
  address  = 南岸学府大道二塘路59号
  struct   = 3室2厅1卫
  area     = 85.7㎡
  floor    = 低层(共27层)
  maketime = 2018年建造
  price    = 16914元/㎡
--- 第 498 行 ---
  title    = 恒大照母山 精装3房 内圈位置 户型方正 拎包即住 诚意出售
  house    = 恒大照母山山水城
  address  = 渝北照母山恒康路9号
  struct   = 3室2厅1卫
  area     = 90.2㎡
  floor    = 低层(共32层)
  maketime = 2015年建造
  price    = 15530元/㎡
